# Хакатон-2026. Цифриум. Машинное обучение

## Команда "Три сигма"

Задача: Создание прогнозной модели успешности студентов на основе анализа данных LMS Цифриум для выявления паттернов оттока и повышения доходимости онлайн-курсов (Датасет - https://drive.google.com/drive/folders/1RnH3yATWhq7IsMx7DMU7glnwJUEMtw_4)


В данном ноутбуке содержатся основые этапы работы над задачей и ключевые инсайты

# 1. Предобработка данных о поведении студентов, анализ датасета   

#### 1) Формирование единого датасета по четырем модулям для анализа переходов между модулями
Шаги этапа (подробнее: https://github.com/mariyakukhtinova-sys/hackaton_mipt_ds_25/blob/master/modules/heuristic_status_modules_1_3.ipynb):

- загрузка данных из модулей `stats__module_1.csv` ... `stats__module_4.csv`;
- очистка данных: удаление строк без `user_id` и/или `course_id`;
- исследование дублей и удаление в `module 1`;
- приведение таблицы к единой схеме;
- сбор общего датасета по всем модулям;
- проверка корректности статуса студента по бизнес-правилам из документации;
- поиск взаимосвязей для определения действительного статуса студента (за исключением 4 модуля - для него не вычисляем статус "переведен/нет");
- подсчет корреляции с реальным таргетом (модули 1 и 2).

Ключевые моменты:

- Выявление базовой единицы наблюдения “пользователь+курс”
- Выявление неточности статуса обучающегося в исходных сырых таблицах (переведен / не переведен на следующий модуль)
- Фокусировка на метрике "сдал промежуточную аттестацию" (pa_passed)

#### 2) Формирование витрины данных
Шаги этапа (подробнее: https://github.com/mariyakukhtinova-sys/hackaton_mipt_ds_25/blob/master/dataset_cleaning.ipynb):

- загрузка данных из таблиц (за исключением модульных);
- проведены проверки по дубликатам, ключам, связности, базовым временным аномалиям и покрытию признаков по модулям;
- учтён разделитель тысяч `,` внутри числовых значений;
- таблицы приведены к типам, удобным для анализа: даты распарсены, числовые поля приведены к nullable-типам, категориальные поля выделены отдельно;
- формирование витрины для фича-инжиниринга (`./artifacts/user_course_modeling_base.csv`)

Ключевые моменты:

- Выведена верная гранулярность — `user-course`, а не просто `user`;
- Часть признаков необходимо восстанавливать через уроки, группы, тренинги и типы ресурсов (а не из сырых данных);
- `modules/status_modules_complete.csv` хорошо задаёт population по всем четырём модулям, но не всегда напрямую стыкуется с `users_courses`, поэтому для итоговой витрины нужен смешанный способ обогащения;
- Перед построением итоговой витрины из событийных таблиц удаляются точные дубли, чтобы не завышать счётчики активности, просмотров, ответов и периодов доступа;
- При анализе пропусков `module == 4` исключается из `missingness_df`, потому что его текущая поставка неполная и без такого исключения таблица пропусков становится менее интерпретируемой.


# 2. Feature Engineering   

Шаги этапа (подробнее: https://github.com/mariyakukhtinova-sys/hackaton_mipt_ds_25/blob/master/pizhurin/feature_engineerin_pin.ipynb):

- заполнение пропусков;
- удаление пропусков, сильно коррелирующих с целевой переменной (во избежание утечки данных);
- разбиение данных на обучающую, валидационную и тестовую выборки (60 / 20 / 20);
- определение наиболее важных признаков (CatBoost)
- построение предсказания в виде ансамбля простых решающих деревьев 

# 3. Процесс разработки прогностической модели отчисления студентов  

Шаги этапа (подробнее: https://github.com/mariyakukhtinova-sys/hackaton_mipt_ds_25/blob/master/Feature%20Selection%20v2.ipynb):

примечание: данный ноутбук включает часть анализа из других ноутбуков, некоторые гипотезы

- проработка модульных таблиц - фиксация даты экватора, удаление логов от даты экватора 
- прогнозирование итогового статуса студента (перведен/нет) на основе цифрового следа до наступления экватора
- добавление метрики "вовлеченности и ритма успеваемости"
- анализ динамики активности (индекс выгорания) - снижение темпа обучения
- анализ паттернов расписания - учет активности в ночные часы и в выходные 
- формирование словаря признаков
- смена парадигмы: отход от прогнозирования на основе данных до момента экватора

Ключевые моменты:

- Данные из 1 модуля, в отличие от данных 2 и 3 модуля, содержат много шума от активности студентов-новичков
- Модели хуже обучаются на сдвоенных модулях, поэтому необходимо производить обучение на 2 или 3 модуле
- Обучение модели исключительно на профиле успеваемости не дает хорошего результата
- Прирост метрики ROC-AUC с добавлением признака "вовлеченности"
- Плавное снижение темпа обучения является одной из ключевых метрик
- фундаментальный предел: прогнозировать точный исход курса на этапе его экватора с высокой точностью невозможно из-за неустранимой ошибки (Irreducible Error)

## Дальнейший план развития
Текущая реализация (Lift Analysis) в некоторой степени решает коммерческую задачу приоритизации работы кураторов. Однако с математической точки зрения бинарная классификация на статичном срезе данных имеет предел точности (Irreducible Error).

Для пробития текущего "стеклянного потолка" метрик и перехода от статического скоринга к динамическому мониторингу, я планирую внедрить два продвинутых архитектурных подхода.

### Направление 1: Оптимизация признакового пространства (Sequential Feature Selection)
Подход "Kitchen Sink" (использование всех сгенерированных признаков) делает модель уязвимой к проклятию размерности и мультиколлинеарности. Несмотря на устойчивость градиентного бустинга, избыточные признаки создают информационный шум. Анализ SHAP-значений показал глобальную важность переменных, но он не учитывает синергию признаков при построении конкретных деревьев.

План реализации: Я планирую провести строгий алгоритмический отбор признаков методом прямого прохода (Forward Sequential Feature Selection).

Итеративное добавление переменных в модель с замером ROC-AUC на скрытой валидационной выборке.
Построение кривой размерности для нахождения истинного математического оптимума.
Ожидаемый результат: Предварительный анализ показывает, что оптимум вероятнее всего достигается на компактном ядре из 13–20 признаков. Сокращение размерности защитит модель от переобучения. Ожидается, что фундаментальными драйверами предсказания останутся:

Социальный контекст (SOC_peer_activity_ratio) — отклонение от нормы группы.
Скорость онбординга (LMS_days_to_first_action) — задержка старта.
Физика обучения (Velocity_W2_zscore) — динамика изменения темпа.
Микро-паттерны (SEQ_stubbornness_ratio) — неэффективное зацикливание на задачах.

### Направление 2: Анализ выживаемости (Time-to-Event Analysis)
Фундаментальный недостаток бинарной классификации заключается в потере информации о времени. Для текущей модели студент, отчисленный на 5-й день, и студент, отчисленный за день до конца курса, выглядят одинаково (класс 0).

План реализации: Я перейду от предсказания вероятности отчисления к предсказанию времени до отчисления. Задачу необходимо переформулировать в терминах анализа выживаемости (Survival Analysis), используя модели Random Survival Forest (RSF) или Cox Proportional Hazards.

Таргет: Целевая переменная трансформируется в кортеж (Event, Duration), где Event — факт отчисления, а Duration — количество дней активности.
Цензурирование (Right-Censoring): Студенты, успешно завершившие курс, будут помечены как цензурированные данные. Их "время жизни" будет жестко ограничено официальной датой завершения модуля для исключения артефактов данных (остаточных кликов после экзамена).
Ожидаемый результат: Модель начнет прогнозировать индивидуальную кривую выживаемости (Survival Function) для каждого студента. Это позволит отделу сопровождения видеть не просто статичный риск, а динамический прогноз: в какой именно момент времени вероятность отчисления конкретного студента начнет критически возрастать.